# lab.ipynb - Colab 비교 연구 워크벤치

mini GPT를 여러 조건으로 학습하고, loss/생성문/파인튜닝 결과를 비교하는 실험 노트북입니다.

- 기존 `src/` 파일은 수정하지 않습니다.
- 모든 학습과 테스트는 Colab에서 실행합니다.
- 기본 사용 흐름은 `SMOKE`로 점검한 뒤, 한 번에 한 변인만 바꾸며 비교하는 것입니다.


## 0. 빠른 시작

**목적**
- 처음 실행하는 사람이 전체 흐름을 빠르게 통과하도록 합니다.

**수정할 곳**
- 처음에는 `RUN_PRESET = "SMOKE"`를 유지합니다.
- `EXPERIMENTS`는 기본값인 baseline과 sinusoidal position 비교를 그대로 둡니다.
- `PROMPTS`에 생성문을 보고 싶은 시작 문장을 추가합니다.

**출력**
- 실험 manifest 표
- train/val loss 그래프
- 학습 전/후 생성문 비교표
- CSV/PNG/Markdown 결과 파일

**해석**
- 10분 점검: `SMOKE` + 기본 2개 실험이 끝까지 도는지 확인합니다.
- 첫 비교: learned position과 sinusoidal position의 val loss, 생성문, 학습 속도를 비교합니다.
- 첫 보고서: 마지막 요약 Markdown과 저장된 그래프를 보고서에 옮깁니다.


## 0.1 실험 원칙

**목적**
- 비교 실험의 기준을 고정해 결과 해석이 흐려지지 않게 합니다.

**수정할 곳**
- 새 실험을 추가할 때는 `make_experiment("run_name", {"바꿀_값": 값})` 형식을 사용합니다.

**출력**
- 실험 전 manifest에서 baseline 대비 바뀐 값이 표시됩니다.

**해석**
- baseline을 먼저 완주합니다.
- 한 번에 한 변인만 바꿉니다.
- 좋은 후보만 조합 실험으로 넘깁니다.
- 최종 후보는 seed 또는 step 수를 바꿔 다시 확인합니다.


## 1. Colab 환경 설정과 저장소 준비

**목적**
- GitHub 저장소를 Colab에 clone하고, 프로젝트 모듈을 import할 수 있게 준비합니다.

**수정할 곳**
- `REPO_URL`: 저장소 주소를 미리 적어두면 입력 과정을 줄일 수 있습니다.
- `BRANCH`: 특정 브랜치를 쓸 때만 입력합니다.
- `USE_GOOGLE_DRIVE`: 긴 실험 결과를 보존하려면 `True`로 바꿉니다.

**출력**
- `Repo:` 프로젝트 경로
- `Artifacts:` 결과 저장 경로

**해석**
- `Artifacts` 경로 아래에 CSV, PNG, Markdown 결과가 저장됩니다.
- Colab 런타임이 끊길 수 있는 긴 실험은 Google Drive 저장을 권장합니다.


In [ ]:
# Colab 전용 환경 설정
import os
import subprocess
import sys
import time
from pathlib import Path

RUN_STARTED_AT = time.strftime("%Y%m%d-%H%M%S")
IN_COLAB = "google.colab" in sys.modules

# 필요하면 값을 직접 채워두세요. 비워두면 실행 시 입력창이 뜹니다.
REPO_URL = ""  # @param {type:"string"}
BRANCH = ""  # @param {type:"string"}
INSTALL_REQUIREMENTS = True  # @param {type:"boolean"}
USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_OUTPUT_DIR = "gpt-lab-runs"  # @param {type:"string"}


def normalize_github_url(url: str) -> str:
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if IN_COLAB:
    from getpass import getpass

    repo_input = REPO_URL.strip() or input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): ")
    repo_url = normalize_github_url(repo_input)
    token = getpass("GitHub Personal Access Token (Private 저장소면 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        clone_cmd = ["git", "clone"]
        if BRANCH.strip():
            clone_cmd += ["--branch", BRANCH.strip()]
        clone_cmd += [clone_url, str(repo_dir)]
        subprocess.run(clone_cmd, check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()
    print("로컬 미리보기 모드입니다. 이 노트북의 학습/테스트 실행은 Colab에서 진행하세요.")

sys.path.insert(0, str(repo_dir / "src"))

if INSTALL_REQUIREMENTS:
    req_path = repo_dir / "requirements.txt"
    if req_path.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_path)], check=False)

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_DIR = Path("/content/drive/MyDrive") / DRIVE_OUTPUT_DIR / RUN_STARTED_AT
else:
    ARTIFACT_DIR = repo_dir / "checkpoints" / "lab" / RUN_STARTED_AT

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo:", repo_dir)
print("Artifacts:", ARTIFACT_DIR)


## 2. 공통 import, GPU 확인, seed 고정

**목적**
- 실험에 필요한 라이브러리와 프로젝트 모듈을 불러오고 실행 환경을 확인합니다.

**수정할 곳**
- `GLOBAL_SEED`: seed 민감도를 보려면 42, 43, 44처럼 바꿔 반복 실행합니다.

**출력**
- 실행 장치, GPU 이름, PyTorch 버전

**해석**
- `Device: cuda`가 나오면 GPU 학습입니다.
- `Device: cpu`가 나오면 Colab 런타임 유형을 GPU로 바꿉니다.


In [ ]:
import copy
import csv
import json
import math
import random
import statistics
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML

from bpe import BPETokenizer
from dataset import create_dataloader
from attention import MultiHeadAttention
from model import LayerNorm
from train import calc_loss_loader, generate
from finetune import ReviewSentimentDataset, GPTForSequenceClassification, train_epoch_sentiment, evaluate_sentiment


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)


## 3. NSMC 데이터 준비와 미리보기

**목적**
- NSMC 데이터를 내려받고 LM/감성분류 파일을 준비합니다.

**수정할 곳**
- 보통 수정하지 않습니다.

**출력**
- 데이터 파일 존재 여부와 크기
- LM train corpus 미리보기

**해석**
- 모든 `nsmc_*` 파일이 `exists=True`여야 다음 단계로 진행할 수 있습니다.
- 미리보기 문장이 깨지면 데이터 다운로드 또는 인코딩을 확인합니다.


In [ ]:
import download_data

DATA_DIR = repo_dir / "data"
DATA_DIR.mkdir(exist_ok=True)

try:
    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", repr(e))
    print("이미 data/ 파일이 있다면 아래 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = DATA_DIR / "nsmc_lm_train.txt"
LM_VAL_PATH = DATA_DIR / "nsmc_lm_val.txt"
SENTIMENT_TRAIN_PATH = DATA_DIR / "nsmc_sentiment_train.jsonl"
SENTIMENT_VAL_PATH = DATA_DIR / "nsmc_sentiment_val.jsonl"
SENTIMENT_TEST_PATH = DATA_DIR / "nsmc_sentiment_test.jsonl"

for path in [LM_TRAIN_PATH, LM_VAL_PATH, SENTIMENT_TRAIN_PATH, SENTIMENT_VAL_PATH, SENTIMENT_TEST_PATH]:
    print(path.name, "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else 0)

train_corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("LM train chars:", len(train_corpus))
print("LM val chars:", len(val_corpus))
print("\n미리보기:")
print(train_corpus[:500])


## 4. 프로젝트 테스트 셀

**목적**
- 기존 과제 구현이 정상인지 Colab에서 확인합니다.

**수정할 곳**
- `RUN_PROJECT_TESTS=True`로 바꾸면 전체 테스트를 실행합니다.
- 특정 파일만 보고 싶으면 `run_pytest("tests/test_model.py")`처럼 호출합니다.

**출력**
- pytest 결과와 return code

**해석**
- return code가 0이면 기존 구현 검증 통과입니다.
- 학습 실험이 이상하면 먼저 이 셀로 코드 구현 문제인지 확인합니다.


In [ ]:
# 필요할 때 True로 바꾸고 실행하세요.
RUN_PROJECT_TESTS = False  # @param {type:"boolean"}


def run_pytest(target: str = "tests/") -> int:
    cmd = [sys.executable, "-m", "pytest", target, "-q"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print("return code:", result.returncode)
    return result.returncode


if RUN_PROJECT_TESTS:
    run_pytest("tests/")
else:
    print("프로젝트 테스트를 실행하려면 RUN_PROJECT_TESTS=True로 바꾸고 이 셀을 실행하세요.")


## 5. 실험 프리셋과 사용자 수정 영역

**목적**
- 실험 규모, baseline 설정, 비교할 실험 목록, 생성 prompt를 정의합니다.

**수정할 곳**
- `RUN_PRESET`: `SMOKE`, `LIGHT`, `CORE`, `LONG` 중 선택합니다.
- `BASE_EXPERIMENT`: 모든 실험의 기본값입니다.
- `EXPERIMENTS`: 실제로 실행할 비교 실험 목록입니다.
- `PROMPTS`: 학습 전/후 생성문 비교에 사용할 문장입니다.
- `GENERATION_PRESETS`: greedy/balanced/creative decoding 설정입니다.

**출력**
- 선택한 preset과 실험 목록

**해석**
- 처음에는 기본 `EXPERIMENTS`로 learned vs sinusoidal position만 비교합니다.
- 새 실험은 baseline 대비 바뀐 값이 적을수록 해석하기 쉽습니다.


In [ ]:
RUN_PRESET = "SMOKE"  # @param ["SMOKE", "LIGHT", "CORE", "LONG"]

PRESETS = {
    "SMOKE": {
        "tokenizer_vocab_size": 300,
        "tokenizer_train_chars": 8_000,
        "lm_train_chars": 12_000,
        "lm_val_chars": 4_000,
        "context_length": 32,
        "batch_size": 8,
        "max_steps": 20,
        "eval_every": 5,
        "eval_batches": 2,
        "max_new_tokens": 30,
        "finetune_train_samples": 512,
        "finetune_val_samples": 256,
        "finetune_epochs": 1,
    },
    "LIGHT": {
        "tokenizer_vocab_size": 500,
        "tokenizer_train_chars": 50_000,
        "lm_train_chars": 80_000,
        "lm_val_chars": 16_000,
        "context_length": 64,
        "batch_size": 16,
        "max_steps": 150,
        "eval_every": 25,
        "eval_batches": 5,
        "max_new_tokens": 50,
        "finetune_train_samples": 2_000,
        "finetune_val_samples": 800,
        "finetune_epochs": 2,
    },
    "CORE": {
        "tokenizer_vocab_size": 1_000,
        "tokenizer_train_chars": 200_000,
        "lm_train_chars": 350_000,
        "lm_val_chars": 60_000,
        "context_length": 96,
        "batch_size": 24,
        "max_steps": 600,
        "eval_every": 100,
        "eval_batches": 10,
        "max_new_tokens": 80,
        "finetune_train_samples": 8_000,
        "finetune_val_samples": 2_000,
        "finetune_epochs": 3,
    },
    "LONG": {
        "tokenizer_vocab_size": 1_500,
        "tokenizer_train_chars": 500_000,
        "lm_train_chars": 900_000,
        "lm_val_chars": 120_000,
        "context_length": 128,
        "batch_size": 32,
        "max_steps": 1_500,
        "eval_every": 150,
        "eval_batches": 15,
        "max_new_tokens": 100,
        "finetune_train_samples": 20_000,
        "finetune_val_samples": 4_000,
        "finetune_epochs": 3,
    },
}

preset = PRESETS[RUN_PRESET]

BASE_EXPERIMENT = {
    "seed": GLOBAL_SEED,
    "tokenizer_vocab_size": preset["tokenizer_vocab_size"],
    "tokenizer_train_chars": preset["tokenizer_train_chars"],
    "lm_train_chars": preset["lm_train_chars"],
    "lm_val_chars": preset["lm_val_chars"],
    "add_bos_eos_per_line": False,
    "context_length": preset["context_length"],
    "stride": None,
    "batch_size": preset["batch_size"],
    "emb_dim": 96,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.1,
    "qkv_bias": False,
    "ffn_mult": 4,
    "position_encoding": "learned",   # learned | sinusoidal | none
    "activation": "gelu",             # gelu | relu | silu | tanh
    "optimizer": "adamw",             # adamw | adam | sgd
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "scheduler": "none",              # none | cosine_warmup
    "warmup_steps": 20,
    "max_steps": preset["max_steps"],
    "eval_every": preset["eval_every"],
    "eval_batches": preset["eval_batches"],
    "grad_clip": 1.0,
    "grad_accum_steps": 1,
    "amp": True,
    "save_checkpoint": False,
    "generation_temperature": 0.8,
    "generation_top_k": 40,
    "max_new_tokens": preset["max_new_tokens"],
}

GENERATION_PRESETS = {
    "greedy": {"temperature": 0.0, "top_k": None},
    "balanced": {"temperature": 0.8, "top_k": 40},
    "creative": {"temperature": 1.1, "top_k": 80},
}


def make_experiment(name: str, overrides: dict | None = None) -> dict:
    """BASE_EXPERIMENT에서 필요한 값만 덮어써 하나의 run 설정을 만듭니다."""
    config = dict(BASE_EXPERIMENT)
    config["name"] = name
    if overrides:
        config.update(overrides)
    return config


# 비교 실험은 이 리스트에 추가하세요.
EXPERIMENTS = [
    make_experiment("baseline_learned_gelu"),
    make_experiment("sinusoidal_position", {"position_encoding": "sinusoidal"}),
]

PROMPTS = [
    "이 영화는",
    "배우 연기는",
    "정말",
]

print("Preset:", RUN_PRESET)
for exp in EXPERIMENTS:
    print(exp["name"], "| pos=", exp["position_encoding"], "act=", exp["activation"], "lr=", exp["learning_rate"])


## 6. 실험용 모델 옵션

**목적**
- 기존 `src/`를 수정하지 않고 위치 임베딩과 활성화 함수 실험을 가능하게 합니다.

**수정할 곳**
- 보통 수정하지 않습니다.
- 새 activation이나 position encoding 방식을 추가할 때만 이 셀을 수정합니다.

**출력**
- 별도 출력은 없습니다. 이후 실험 실행 함수가 이 클래스를 사용합니다.

**해석**
- `learned`: 위치 벡터를 학습합니다.
- `sinusoidal`: 고정 sin/cos 위치 인코딩을 사용합니다.
- `none`: 위치 정보를 제거해 위치 임베딩의 필요성을 확인합니다.


In [ ]:
class LabInputEmbedding(nn.Module):
    """토큰 임베딩에 선택한 위치 정보를 더해 Transformer 입력을 만듭니다."""

    def __init__(self, vocab_size: int, emb_dim: int, context_length: int, drop_rate: float = 0.1, position_encoding: str = "learned"):
        super().__init__()
        self.emb_dim = emb_dim
        self.context_length = context_length
        self.position_encoding_type = position_encoding
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)

        if position_encoding == "learned":
            # GPT 계열에서 흔히 쓰는 학습형 위치 임베딩입니다.
            self.position_embedding = nn.Embedding(context_length, emb_dim)
        elif position_encoding == "sinusoidal":
            # Transformer 원 논문의 고정 sin/cos 위치 인코딩입니다.
            position = torch.arange(context_length, dtype=torch.float).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, emb_dim, 2, dtype=torch.float) * (-math.log(10000.0) / emb_dim))
            pe = torch.zeros(context_length, emb_dim)
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])
            self.register_buffer("position_encoding", pe)
        elif position_encoding == "none":
            # 위치 정보를 제거해 position encoding의 효과를 비교합니다.
            pass
        else:
            raise ValueError(f"Unknown position_encoding: {position_encoding}")

        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, seq_len = x.shape
        if seq_len > self.context_length:
            raise ValueError(f"seq_len={seq_len} exceeds context_length={self.context_length}")

        token_embeds = self.token_embedding(x)
        if self.position_encoding_type == "learned":
            positions = torch.arange(seq_len, device=x.device)
            embeddings = token_embeds + self.position_embedding(positions)
        elif self.position_encoding_type == "sinusoidal":
            pos = self.position_encoding[:seq_len].to(device=x.device, dtype=token_embeds.dtype)
            embeddings = token_embeds + pos
        else:
            embeddings = token_embeds
        return self.dropout(embeddings)


def make_activation(name: str) -> nn.Module:
    """문자열 설정을 실제 activation 모듈로 변환합니다."""
    name = name.lower()
    if name == "gelu":
        return nn.GELU()
    if name == "relu":
        return nn.ReLU()
    if name == "silu":
        return nn.SiLU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError(f"Unknown activation: {name}")


class LabFeedForward(nn.Module):
    """Transformer block 내부 MLP입니다. ffn_mult로 hidden 확장 배율을 바꿉니다."""

    def __init__(self, d_model: int, dropout: float = 0.1, mult: int = 4, activation: str = "gelu"):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, mult * d_model),
            make_activation(activation),
            nn.Linear(mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class LabTransformerBlock(nn.Module):
    """Attention과 FeedForward를 residual connection으로 묶은 GPT block입니다."""

    def __init__(self, d_model: int, n_heads: int, drop_rate: float = 0.1, qkv_bias: bool = False, ffn_mult: int = 4, activation: str = "gelu"):
        super().__init__()
        self.attention = MultiHeadAttention(d_model=d_model, n_heads=n_heads, drop_rate=drop_rate, qkv_bias=qkv_bias)
        self.ffn = LabFeedForward(d_model=d_model, dropout=drop_rate, mult=ffn_mult, activation=activation)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)

    def forward(self, x: torch.Tensor, causal_mask: bool = True) -> torch.Tensor:
        x = x + self.attention(self.norm1(x), causal_mask=causal_mask)
        x = x + self.ffn(self.norm2(x))
        return x


class LabGPTModel(nn.Module):
    """실험 옵션을 반영한 mini GPT 언어모델입니다."""

    def __init__(self, config: dict):
        super().__init__()
        self.config = dict(config)
        self.embedding = LabInputEmbedding(
            vocab_size=config["vocab_size"],
            emb_dim=config["emb_dim"],
            context_length=config["context_length"],
            drop_rate=config["drop_rate"],
            position_encoding=config.get("position_encoding", "learned"),
        )
        self.blocks = nn.Sequential(*[
            LabTransformerBlock(
                d_model=config["emb_dim"],
                n_heads=config["n_heads"],
                drop_rate=config["drop_rate"],
                qkv_bias=config["qkv_bias"],
                ffn_mult=config.get("ffn_mult", 4),
                activation=config.get("activation", "gelu"),
            )
            for _ in range(config["n_layers"])
        ])
        self.final_norm = LayerNorm(config["emb_dim"])
        self.lm_head = nn.Linear(config["emb_dim"], config["vocab_size"], bias=False)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        x = self.embedding(idx)
        x = self.blocks(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return loss, logits


## 7. 로깅, 표, 그래프, export 유틸리티

**목적**
- 실험 결과를 화면 표, 그래프, CSV, PNG, Markdown으로 정리합니다.

**수정할 곳**
- `FIG_DPI`: 저장 그림 해상도.
- `TABLE_MAX_TEXT`: 화면 표에서 긴 텍스트를 몇 글자까지 보여줄지 정합니다.

**출력**
- 화면: 가로 스크롤이 되는 HTML 표.
- 파일: `lm_summary.csv`, `lm_history.csv`, `generation_samples.csv`, `finetune_*.csv`.
- 그림: loss/perplexity/gap/efficiency 그래프 PNG.

**해석**
- 화면 표는 빠른 확인용입니다.
- CSV는 보고서 표, 스프레드시트, 추가 분석용입니다.
- Markdown 요약은 보고서 초안으로 바로 옮기기 위한 출력입니다.


## 7.1 결과 데이터 사전

**목적**
- 노트북이 출력하는 표와 CSV 컬럼의 의미를 정리합니다.

### LM 실험 요약: `lm_summary.csv`
| 컬럼 | 의미 | 해석 |
| --- | --- | --- |
| `name` | 실험 이름 | `EXPERIMENTS`에서 지정한 run 이름입니다. |
| `position` | 위치 정보 방식 | `learned`, `sinusoidal`, `none` 중 하나입니다. |
| `activation` | FFN 활성화 함수 | `gelu`, `relu`, `silu`, `tanh` 등입니다. |
| `optimizer` | 최적화 함수 | `adamw`, `adam`, `sgd` 등입니다. |
| `scheduler` | learning rate 스케줄 | `none`, `cosine_warmup` 등입니다. |
| `params` | 전체 파라미터 수 | 모델 크기입니다. 작을수록 빠르고, 클수록 표현력이 커질 수 있습니다. |
| `best_val_loss` | 가장 낮은 검증 loss | LM 성능의 핵심 비교 지표입니다. 낮을수록 좋습니다. |
| `delta_vs_baseline` | baseline 대비 loss 차이 | 음수면 baseline보다 좋고, 양수면 나쁩니다. |
| `best_ppl` | best val loss의 perplexity | 낮을수록 다음 토큰 예측이 더 안정적입니다. |
| `final_train_loss` | 마지막 평가 시점 train loss | 학습 데이터에 대한 최종 적합도입니다. |
| `runtime_sec`, `runtime` | 학습 시간 | 실험 비용을 비교합니다. |
| `tokens_per_sec` | 초당 처리 토큰 수 | 학습 속도입니다. 높을수록 빠릅니다. |

### LM 학습 과정: `lm_history.csv`
| 컬럼 | 의미 | 해석 |
| --- | --- | --- |
| `run` | 실험 이름 | 어느 실험의 기록인지 구분합니다. |
| `step` | optimizer step | x축으로 사용합니다. |
| `train_loss` | 학습 데이터 loss | 내려가면 학습 데이터를 더 잘 맞추는 중입니다. |
| `val_loss` | 검증 데이터 loss | 일반화 성능의 핵심 지표입니다. |
| `train_ppl`, `val_ppl` | train/val perplexity | loss를 직관적으로 바꾼 값입니다. 낮을수록 좋습니다. |
| `val_train_gap` | `val_loss - train_loss` | 커지면 과적합 가능성이 있습니다. |
| `lr` | 현재 learning rate | scheduler 동작 확인용입니다. |
| `elapsed_sec` | 누적 실행 시간 | step별 시간 흐름입니다. |

### 생성문 비교: `generation_samples.csv`
| 컬럼 | 의미 | 해석 |
| --- | --- | --- |
| `run` | 실험 이름 | 어떤 모델의 생성 결과인지 구분합니다. |
| `phase` | `before` 또는 `after` | 학습 전/후 변화를 비교합니다. |
| `decoding` | 생성 방식 | `greedy`는 안정적, `creative`는 다양하지만 흔들릴 수 있습니다. |
| `prompt` | 입력 문장 | 모델에게 넣은 시작 문장입니다. |
| `text` | 생성 결과 | 정성 평가용입니다. loss와 함께 해석해야 합니다. |

### 파인튜닝 요약: `finetune_summary.csv`
| 컬럼 | 의미 | 해석 |
| --- | --- | --- |
| `name` | 파인튜닝 실험 이름 | freeze mode나 lr 차이를 구분합니다. |
| `base` | 사용한 LM backbone | 어떤 사전학습 run에서 출발했는지 나타냅니다. |
| `freeze` | 튜닝 범위 | `classifier_only`, `last_block_plus_head`, `full_finetune`. |
| `trainable_params` | 학습되는 파라미터 수 | 클수록 오래 걸리고 과적합 위험도 커질 수 있습니다. |
| `best_val_acc` | 최고 검증 정확도 | validation 기준 선택 지표입니다. |
| `test_acc` | test 정확도 | 최종 일반화 성능입니다. |
| `precision` | 긍정 예측의 정밀도 | 긍정이라고 한 것 중 실제 긍정 비율입니다. |
| `recall` | 긍정 샘플 재현율 | 실제 긍정 중 맞춘 비율입니다. |
| `f1` | precision/recall 조화 평균 | class 불균형이나 tradeoff를 함께 봅니다. |
| `fp`, `fn` | false positive/false negative | 어느 방향으로 틀리는지 봅니다. |

### 파인튜닝 예측: `finetune_predictions.csv`
| 컬럼 | 의미 | 해석 |
| --- | --- | --- |
| `text` | 리뷰 문장 | 실제 입력입니다. |
| `label` | 정답 | `0=부정`, `1=긍정`. |
| `pred` | 모델 예측 | `0=부정`, `1=긍정`. |
| `confidence` | 예측 확신도 | 높게 틀린 예시는 모델의 약점을 보여줍니다. |
| `correct` | 정답 여부 | `1=정답`, `0=오답`. |
| `error_type` | 오답 유형 | `true_0_pred_1`, `true_1_pred_0` 등입니다. |


In [ ]:
RUNS = []
FINETUNE_RUNS = []
FIG_DPI = 160
TABLE_MAX_TEXT = 180

import html


def count_parameters(model: nn.Module, trainable_only: bool = False) -> int:
    """모델 파라미터 수를 계산합니다. trainable_only=True면 학습되는 파라미터만 셉니다."""
    params = model.parameters()
    if trainable_only:
        return sum(p.numel() for p in params if p.requires_grad)
    return sum(p.numel() for p in params)


def estimate_model_parameters(config: dict) -> int:
    """실험 전 manifest에서 모델 크기를 미리 보여주기 위한 가벼운 추정 함수입니다."""
    cfg = dict(config)
    cfg["vocab_size"] = config.get("actual_vocab_size", config.get("tokenizer_vocab_size", 300))
    model_cfg = model_config_from_experiment(cfg) if "model_config_from_experiment" in globals() else {
        "vocab_size": cfg["vocab_size"],
        "context_length": cfg["context_length"],
        "emb_dim": cfg["emb_dim"],
        "n_heads": cfg["n_heads"],
        "n_layers": cfg["n_layers"],
        "drop_rate": cfg["drop_rate"],
        "qkv_bias": cfg["qkv_bias"],
        "ffn_mult": cfg.get("ffn_mult", 4),
        "position_encoding": cfg.get("position_encoding", "learned"),
        "activation": cfg.get("activation", "gelu"),
    }
    was_device = globals().get("device", torch.device("cpu"))
    model = LabGPTModel(model_cfg).to("cpu")
    n_params = count_parameters(model)
    del model
    if was_device.type == "cuda":
        torch.cuda.empty_cache()
    return n_params


def perplexity(loss: float) -> float:
    """cross entropy loss를 perplexity로 변환합니다."""
    if loss is None or math.isnan(loss):
        return float("nan")
    return float(math.exp(min(20.0, loss)))


def format_seconds(seconds: float) -> str:
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m}m {s}s"
    if m:
        return f"{m}m {s}s"
    return f"{s}s"


def md_escape(value) -> str:
    """Markdown 파일로 내보낼 때 셀 안의 특수문자와 줄바꿈을 안전하게 바꿉니다."""
    text = "" if value is None else str(value)
    return text.replace("|", "\\|").replace(chr(10), "<br>")


def html_cell(value, max_text: int = TABLE_MAX_TEXT) -> str:
    """화면 표시용 HTML 셀을 만듭니다. 긴 텍스트는 줄여서 표가 무너지지 않게 합니다."""
    text = "" if value is None else str(value)
    if len(text) > max_text:
        text = text[:max_text] + "..."
    return html.escape(text).replace(chr(10), "<br>")


def display_table(rows: list[dict], columns: list[tuple[str, str]], title: str | None = None) -> None:
    """Colab에서 읽기 쉬운 HTML 표로 표시합니다. CSV export는 원본 값을 그대로 저장합니다."""
    if not rows:
        title_html = f"<h3>{html.escape(title)}</h3>" if title else ""
        display(HTML(title_html + "<p><em>표시할 결과가 없습니다.</em></p>"))
        return

    style = """
    <style>
    .lab-table-wrap { overflow-x: auto; max-width: 100%; margin: 0.5rem 0 1rem 0; }
    table.lab-table { border-collapse: collapse; font-size: 13px; min-width: 760px; }
    table.lab-table th, table.lab-table td { border: 1px solid #ddd; padding: 6px 8px; vertical-align: top; }
    table.lab-table th { background: #f6f8fa; font-weight: 600; white-space: nowrap; }
    table.lab-table td { max-width: 360px; white-space: normal; word-break: break-word; }
    </style>
    """
    title_html = f"<h3>{html.escape(title)}</h3>" if title else ""
    header = "".join(f"<th>{html.escape(label)}</th>" for _, label in columns)
    body_rows = []
    for row in rows:
        cells = "".join(f"<td>{html_cell(row.get(key, ''))}</td>" for key, _ in columns)
        body_rows.append(f"<tr>{cells}</tr>")
    table_html = f"<div class='lab-table-wrap'><table class='lab-table'><thead><tr>{header}</tr></thead><tbody>{''.join(body_rows)}</tbody></table></div>"
    display(HTML(style + title_html + table_html))

def write_csv(path: Path, rows: list[dict], columns: list[str] | None = None) -> None:
    """보고서와 스프레드시트에서 쓰기 쉬운 CSV 파일을 저장합니다."""
    if columns is None:
        keys = []
        for row in rows:
            for key in row.keys():
                if key not in keys:
                    keys.append(key)
        columns = keys
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, "") for key in columns})


def save_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + chr(10))


def save_current_figure(filename: str) -> Path:
    """현재 matplotlib figure를 artifact 폴더에 저장합니다."""
    path = ARTIFACT_DIR / filename
    plt.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    print("saved figure:", path)
    return path


def serializable_run(run: dict) -> dict:
    return {
        "summary": run.get("summary", {}),
        "history": run.get("history", []),
        "samples": run.get("samples", []),
        "config": run.get("config", {}),
    }


def lm_summary_rows(runs: list[dict]) -> list[dict]:
    rows = []
    if not runs:
        return rows
    baseline = runs[0]["summary"]
    baseline_loss = baseline["best_val_loss"]
    for run in runs:
        s = run["summary"]
        delta = s["best_val_loss"] - baseline_loss
        rows.append({
            "name": s["name"],
            "position": s["position_encoding"],
            "activation": s["activation"],
            "optimizer": s["optimizer"],
            "scheduler": s["scheduler"],
            "params": s["total_params"],
            "best_val_loss": s["best_val_loss"],
            "delta_vs_baseline": delta,
            "best_ppl": perplexity(s["best_val_loss"]),
            "final_train_loss": s["final_train_loss"],
            "runtime_sec": s["runtime_sec"],
            "runtime": s["runtime"],
            "tokens_per_sec": s["tokens_per_sec"],
        })
    return rows


def display_lm_summary(runs: list[dict]) -> None:
    rows = []
    for row in lm_summary_rows(runs):
        rows.append({
            "name": row["name"],
            "position": row["position"],
            "activation": row["activation"],
            "params": f"{row['params']:,}",
            "best_val_loss": f"{row['best_val_loss']:.3f}",
            "delta": f"{row['delta_vs_baseline']:+.3f}",
            "best_ppl": f"{row['best_ppl']:.1f}",
            "runtime": row["runtime"],
            "tokens/sec": f"{row['tokens_per_sec']:.0f}",
        })
    display_table(
        rows,
        [("name", "run"), ("position", "position"), ("activation", "activation"), ("params", "params"), ("best_val_loss", "best val loss"), ("delta", "delta"), ("best_ppl", "best ppl"), ("runtime", "runtime"), ("tokens/sec", "tokens/sec")],
        title="LM 실험 요약",
    )


def lm_history_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        for h in run.get("history", []):
            rows.append({
                "run": run["summary"]["name"],
                "step": h.get("step"),
                "train_loss": h.get("train_loss"),
                "val_loss": h.get("val_loss"),
                "train_ppl": perplexity(h.get("train_loss")),
                "val_ppl": perplexity(h.get("val_loss")),
                "val_train_gap": h.get("val_loss") - h.get("train_loss"),
                "lr": h.get("lr"),
                "elapsed_sec": h.get("elapsed_sec"),
            })
    return rows


def generation_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        for sample in run.get("samples", []):
            rows.append({
                "run": run["summary"]["name"],
                "phase": sample.get("phase", "after"),
                "decoding": sample.get("decoding", "balanced"),
                "prompt": sample.get("prompt", ""),
                "text": sample.get("text", ""),
            })
    return rows


def export_lm_artifacts(runs: list[dict]) -> None:
    """LM 실험 결과를 JSONL과 CSV로 저장합니다."""
    save_jsonl(ARTIFACT_DIR / "lm_runs.jsonl", [serializable_run(run) for run in runs])
    write_csv(ARTIFACT_DIR / "lm_summary.csv", lm_summary_rows(runs))
    write_csv(ARTIFACT_DIR / "lm_history.csv", lm_history_rows(runs))
    write_csv(ARTIFACT_DIR / "generation_samples.csv", generation_rows(runs))
    print("saved LM artifacts:", ARTIFACT_DIR)


def save_lm_runs(runs: list[dict]) -> Path:
    export_lm_artifacts(runs)
    return ARTIFACT_DIR / "lm_runs.jsonl"


def plot_lm_histories(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    plt.figure(figsize=(12, 5))
    for run in runs:
        history = run["history"]
        steps = [h["step"] for h in history]
        train_losses = [h["train_loss"] for h in history]
        val_losses = [h["val_loss"] for h in history]
        plt.plot(steps, train_losses, marker="o", label=f"{run['summary']['name']} train")
        plt.plot(steps, val_losses, marker="x", linestyle="--", label=f"{run['summary']['name']} val")
    plt.xlabel("optimizer step")
    plt.ylabel("loss")
    plt.title("Pretraining loss comparison")
    plt.legend()
    plt.grid(alpha=0.25)
    if save:
        save_current_figure("lm_loss.png")
    plt.show()


def plot_lm_perplexity(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    plt.figure(figsize=(12, 5))
    for run in runs:
        history = run["history"]
        steps = [h["step"] for h in history]
        val_ppl = [perplexity(h["val_loss"]) for h in history]
        plt.plot(steps, val_ppl, marker="o", label=run["summary"]["name"])
    plt.xlabel("optimizer step")
    plt.ylabel("validation perplexity")
    plt.title("Validation perplexity comparison")
    plt.legend()
    plt.grid(alpha=0.25)
    if save:
        save_current_figure("lm_perplexity.png")
    plt.show()


def plot_lm_gap(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    plt.figure(figsize=(12, 5))
    for run in runs:
        history = run["history"]
        steps = [h["step"] for h in history]
        gaps = [h["val_loss"] - h["train_loss"] for h in history]
        plt.plot(steps, gaps, marker="o", label=run["summary"]["name"])
    plt.axhline(0, color="black", linewidth=1)
    plt.xlabel("optimizer step")
    plt.ylabel("val loss - train loss")
    plt.title("Generalization gap")
    plt.legend()
    plt.grid(alpha=0.25)
    if save:
        save_current_figure("lm_val_train_gap.png")
    plt.show()


def plot_efficiency_scatter(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    plt.figure(figsize=(7, 5))
    for run in runs:
        s = run["summary"]
        plt.scatter(s["total_params"], s["best_val_loss"], s=80)
        plt.text(s["total_params"], s["best_val_loss"], " " + s["name"], va="center")
    plt.xlabel("parameter count")
    plt.ylabel("best validation loss")
    plt.title("Quality vs model size")
    plt.grid(alpha=0.25)
    if save:
        save_current_figure("lm_quality_vs_params.png")
    plt.show()


def plot_runtime_scatter(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    plt.figure(figsize=(7, 5))
    for run in runs:
        s = run["summary"]
        plt.scatter(s["runtime_sec"], s["best_val_loss"], s=80)
        plt.text(s["runtime_sec"], s["best_val_loss"], " " + s["name"], va="center")
    plt.xlabel("runtime seconds")
    plt.ylabel("best validation loss")
    plt.title("Quality vs runtime")
    plt.grid(alpha=0.25)
    if save:
        save_current_figure("lm_quality_vs_runtime.png")
    plt.show()


def config_diff_from_base(config: dict, base: dict = BASE_EXPERIMENT) -> str:
    changed = []
    ignored = {"name"}
    for key in sorted(config.keys()):
        if key in ignored or key not in base:
            continue
        if config[key] != base[key]:
            changed.append(f"{key}={config[key]}")
    return ", ".join(changed) if changed else "baseline"


def display_experiment_manifest(experiments: list[dict]) -> None:
    """학습 전에 run 목록과 baseline 대비 변경점을 확인합니다."""
    rows = []
    for config in experiments:
        try:
            estimated_params = estimate_model_parameters(config)
            params_text = f"{estimated_params:,}"
        except Exception as e:
            params_text = f"estimate failed: {e}"
        rows.append({
            "name": config["name"],
            "changed": config_diff_from_base(config),
            "params": params_text,
            "steps": config["max_steps"],
            "ctx": config["context_length"],
            "batch": config["batch_size"],
            "lr": config["learning_rate"],
            "optimizer": config["optimizer"],
        })
    display_table(rows, [("name", "run"), ("changed", "changed vs baseline"), ("params", "estimated params"), ("steps", "steps"), ("ctx", "ctx"), ("batch", "batch"), ("lr", "lr"), ("optimizer", "optimizer")], title="실험 실행 전 확인")


## 8. 토크나이저와 데이터로더

**목적**
- BPE tokenizer를 학습/캐시하고 LM 학습용 DataLoader를 만듭니다.

**수정할 곳**
- 보통 `BASE_EXPERIMENT`의 tokenizer/data 관련 값을 수정합니다.

**출력**
- tokenizer cache 파일
- train/val token 수와 batch 수

**해석**
- token 수가 너무 적으면 loss와 생성문 비교가 불안정합니다.
- 같은 tokenizer 설정은 캐시되어 다음 실행부터 빨라집니다.


In [ ]:
TOKENIZER_CACHE = {}


def train_or_load_tokenizer(vocab_size: int, corpus: str, train_chars: int) -> BPETokenizer:
    """BPE tokenizer를 학습하거나 data/ 캐시에서 재사용합니다."""
    key = (vocab_size, train_chars)
    if key in TOKENIZER_CACHE:
        return TOKENIZER_CACHE[key]

    cache_path = DATA_DIR / f"vocab_lab_v{vocab_size}_c{train_chars}.json"
    tokenizer = BPETokenizer(vocab_size=vocab_size)
    if cache_path.exists():
        tokenizer.load(cache_path)
        print("Loaded tokenizer:", cache_path)
    else:
        train_text = corpus[:train_chars]
        if not train_text.strip():
            raise ValueError("Tokenizer 학습 corpus가 비어 있습니다. 데이터 준비 셀을 먼저 확인하세요.")
        print(f"Training tokenizer: vocab_size={vocab_size}, chars={len(train_text):,}")
        started = time.time()
        tokenizer.train(train_text)
        tokenizer.save(cache_path)
        print("Saved tokenizer:", cache_path, "elapsed=", format_seconds(time.time() - started))

    TOKENIZER_CACHE[key] = tokenizer
    return tokenizer


def encode_corpus(tokenizer: BPETokenizer, text: str, char_limit: int, add_bos_eos_per_line: bool = False) -> list[int]:
    """텍스트 corpus를 LM 학습용 token id 리스트로 변환합니다."""
    text = text[:char_limit]
    if add_bos_eos_per_line:
        ids = []
        for line in text.splitlines():
            line = line.strip()
            if line:
                ids.extend(tokenizer.encode(line, add_bos_eos=True))
        return ids
    return tokenizer.encode(text)


def build_lm_loaders(config: dict, tokenizer: BPETokenizer):
    """실험 설정으로 train/validation DataLoader와 token 통계를 만듭니다."""
    context_length = config["context_length"]
    train_ids = encode_corpus(tokenizer, train_corpus, config["lm_train_chars"], config.get("add_bos_eos_per_line", False))
    val_ids = encode_corpus(tokenizer, val_corpus, config["lm_val_chars"], config.get("add_bos_eos_per_line", False))

    if len(train_ids) <= context_length + 1:
        raise ValueError(f"train token 수가 너무 적습니다: {len(train_ids)} tokens")
    if len(val_ids) <= context_length + 1:
        split = max(context_length + 2, int(len(train_ids) * 0.1))
        val_ids = train_ids[-split:]
        train_ids = train_ids[:-split]

    stride = config.get("stride") or context_length
    train_loader = create_dataloader(
        train_ids,
        context_length=context_length,
        batch_size=config["batch_size"],
        stride=stride,
        drop_last=True,
        shuffle=True,
        num_workers=0,
    )
    val_loader = create_dataloader(
        val_ids,
        context_length=context_length,
        batch_size=config["batch_size"],
        stride=stride,
        drop_last=False,
        shuffle=False,
        num_workers=0,
    )
    token_stats = {
        "train_tokens": len(train_ids),
        "val_tokens": len(val_ids),
        "train_batches": len(train_loader),
        "val_batches": len(val_loader),
    }
    return train_loader, val_loader, token_stats


## 9. Optimizer, scheduler, 학습 루프

**목적**
- 모델을 만들고 `max_steps` 기준으로 사전학습을 실행합니다.

**수정할 곳**
- `optimizer`, `learning_rate`, `scheduler`, `warmup_steps`, `grad_clip`, `grad_accum_steps`, `amp`

**출력**
- step별 train/val loss history
- best validation loss, runtime, tokens/sec

**해석**
- loss가 거의 내려가지 않으면 learning rate나 데이터 크기를 먼저 확인합니다.
- train loss만 내려가고 val loss가 오르면 과적합 가능성이 큽니다.
- 큰 모델이 작은 모델보다 못하면 step 수, lr, 데이터량이 부족할 수 있습니다.


In [ ]:
def model_config_from_experiment(config: dict) -> dict:
    """실험 설정 dict에서 LabGPTModel이 필요한 값만 추립니다."""
    return {
        "vocab_size": config.get("actual_vocab_size", config["tokenizer_vocab_size"]),
        "context_length": config["context_length"],
        "emb_dim": config["emb_dim"],
        "n_heads": config["n_heads"],
        "n_layers": config["n_layers"],
        "drop_rate": config["drop_rate"],
        "qkv_bias": config["qkv_bias"],
        "ffn_mult": config.get("ffn_mult", 4),
        "position_encoding": config.get("position_encoding", "learned"),
        "activation": config.get("activation", "gelu"),
    }


def build_optimizer(model: nn.Module, config: dict):
    """실험 설정에 따라 optimizer를 만듭니다."""
    name = config.get("optimizer", "adamw").lower()
    lr = config["learning_rate"]
    wd = config.get("weight_decay", 0.0)
    params = [p for p in model.parameters() if p.requires_grad]
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=wd)
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd)
    raise ValueError(f"Unknown optimizer: {name}")


def build_scheduler(optimizer, config: dict):
    """none 또는 warmup+cosine learning rate schedule을 반환합니다."""
    name = config.get("scheduler", "none").lower()
    if name == "none":
        return None
    if name == "cosine_warmup":
        warmup_steps = max(1, int(config.get("warmup_steps", 10)))
        max_steps = max(warmup_steps + 1, int(config["max_steps"]))

        def lr_lambda(step: int):
            # warmup 동안 lr을 선형 증가시키고 이후 cosine으로 천천히 낮춥니다.
            if step < warmup_steps:
                return max(1e-8, (step + 1) / warmup_steps)
            progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
            return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    raise ValueError(f"Unknown scheduler: {name}")


def next_batch(iterator, loader):
    """DataLoader iterator가 끝나면 다시 처음부터 이어서 batch를 꺼냅니다."""
    try:
        return next(iterator), iterator
    except StopIteration:
        iterator = iter(loader)
        return next(iterator), iterator


def evaluate_lm(model: nn.Module, train_loader, val_loader, config: dict) -> tuple[float, float]:
    """일부 batch만 사용해 빠르게 train/val loss를 평가합니다."""
    eval_batches = config.get("eval_batches", 5)
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_batches)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_batches)
    return float(train_loss), float(val_loss)


def train_lm_model(config: dict, tokenizer: BPETokenizer, train_loader, val_loader, token_stats: dict):
    """하나의 LM 실험을 학습하고 history와 summary를 반환합니다."""
    set_seed(config.get("seed", GLOBAL_SEED))
    model_cfg = model_config_from_experiment(config)
    model = LabGPTModel(model_cfg).to(device)
    optimizer = build_optimizer(model, config)
    scheduler = build_scheduler(optimizer, config)

    # AMP는 GPU에서 mixed precision을 써서 속도와 메모리를 아끼는 옵션입니다.
    use_amp = bool(config.get("amp", True)) and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    grad_accum_steps = max(1, int(config.get("grad_accum_steps", 1)))
    max_steps = int(config["max_steps"])
    eval_every = int(config.get("eval_every", 50))
    grad_clip = config.get("grad_clip", None)

    history = []
    started = time.time()
    train_iter = iter(train_loader)

    initial_train_loss, initial_val_loss = evaluate_lm(model, train_loader, val_loader, config)
    history.append({
        "step": 0,
        "train_loss": initial_train_loss,
        "val_loss": initial_val_loss,
        "lr": optimizer.param_groups[0]["lr"],
        "elapsed_sec": 0.0,
    })
    print(f"[{config['name']}] step 0000 | train {initial_train_loss:.3f} | val {initial_val_loss:.3f}")

    for step in range(1, max_steps + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        micro_losses = []

        # grad_accum_steps > 1이면 여러 micro batch의 gradient를 누적한 뒤 한 번 optimizer step을 진행합니다.
        for _ in range(grad_accum_steps):
            (input_batch, target_batch), train_iter = next_batch(train_iter, train_loader)
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)

            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss, _ = model(input_batch, targets=target_batch)
                scaled_loss = loss / grad_accum_steps

            scaler.scale(scaled_loss).backward()
            micro_losses.append(float(loss.detach().cpu()))

        if grad_clip is not None and grad_clip > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()

        if step % eval_every == 0 or step == max_steps:
            train_loss, val_loss = evaluate_lm(model, train_loader, val_loader, config)
            elapsed = time.time() - started
            history.append({
                "step": step,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "lr": optimizer.param_groups[0]["lr"],
                "batch_loss": statistics.mean(micro_losses),
                "elapsed_sec": elapsed,
            })
            print(f"[{config['name']}] step {step:04d} | train {train_loss:.3f} | val {val_loss:.3f} | lr {optimizer.param_groups[0]['lr']:.2e} | {format_seconds(elapsed)}")

    runtime_sec = time.time() - started
    if config.get("save_checkpoint", False):
        ckpt_path = ARTIFACT_DIR / f"{config['name']}_step{max_steps}.pt"
        torch.save({"model_state_dict": model.state_dict(), "config": model_cfg, "experiment": config}, ckpt_path)
        print("checkpoint saved:", ckpt_path)

    tokens_seen = max_steps * config["batch_size"] * config["context_length"] * grad_accum_steps
    best_val_loss = min(h["val_loss"] for h in history)
    final_train_loss = history[-1]["train_loss"]

    summary = {
        "name": config["name"],
        "position_encoding": config.get("position_encoding", "learned"),
        "activation": config.get("activation", "gelu"),
        "optimizer": config.get("optimizer", "adamw"),
        "scheduler": config.get("scheduler", "none"),
        "total_params": count_parameters(model),
        "trainable_params": count_parameters(model, trainable_only=True),
        "best_val_loss": best_val_loss,
        "final_train_loss": final_train_loss,
        "runtime_sec": runtime_sec,
        "runtime": format_seconds(runtime_sec),
        "tokens_seen": tokens_seen,
        "tokens_per_sec": tokens_seen / max(1e-9, runtime_sec),
        **token_stats,
    }
    return model, history, summary


## 10. 생성문 비교와 실험 실행 함수

**목적**
- 학습 전/후 모델에 같은 prompt를 넣어 생성 결과를 비교합니다.

**수정할 곳**
- `PROMPTS`: 비교할 시작 문장
- `GENERATION_PRESETS`: greedy/balanced/creative decoding 값

**출력**
- `before`와 `after` 생성문 비교표
- `generation_samples.csv`

**해석**
- 생성문이 좋아 보여도 val loss가 나쁘면 우연일 수 있습니다.
- 같은 prompt에서 before보다 after가 더 자연스러워지는지 확인합니다.


In [ ]:
def decoding_config(config: dict, preset_name: str = "balanced") -> dict:
    """generation preset과 실험별 generation 설정을 합칩니다."""
    preset_cfg = GENERATION_PRESETS.get(preset_name, GENERATION_PRESETS["balanced"])
    return {
        "temperature": preset_cfg.get("temperature", config.get("generation_temperature", 0.8)),
        "top_k": preset_cfg.get("top_k", config.get("generation_top_k", 40)),
    }


def generate_text(model: nn.Module, tokenizer: BPETokenizer, prompt: str, config: dict, decoding: str = "balanced") -> str:
    """프롬프트를 token id로 바꾸고 모델이 이어 쓴 텍스트를 반환합니다."""
    model.eval()
    encoded = tokenizer.encode(prompt)
    if not encoded:
        encoded = [tokenizer.get_bos_id()]
    idx = torch.tensor(encoded, dtype=torch.long, device=device).unsqueeze(0)
    dec_cfg = decoding_config(config, decoding)
    token_ids = generate(
        model=model,
        idx=idx,
        max_new_tokens=config.get("max_new_tokens", 50),
        context_size=config["context_length"],
        temperature=dec_cfg["temperature"],
        top_k=dec_cfg["top_k"],
    )
    return tokenizer.decode(token_ids.squeeze(0).tolist()).replace(chr(10), " ")


def generate_samples(model: nn.Module, tokenizer: BPETokenizer, config: dict, prompts: list[str], phase: str, decoding_names: list[str] | None = None) -> list[dict]:
    """여러 prompt와 decoding preset에 대한 생성 결과를 표 형태 데이터로 만듭니다."""
    decoding_names = decoding_names or ["balanced"]
    samples = []
    for decoding in decoding_names:
        for prompt in prompts:
            try:
                text = generate_text(model, tokenizer, prompt, config, decoding=decoding)
            except Exception as e:
                text = f"[generation failed: {e}]"
            samples.append({"phase": phase, "decoding": decoding, "prompt": prompt, "text": text})
    return samples


def build_fresh_lm_model(config: dict, tokenizer: BPETokenizer) -> nn.Module:
    """같은 seed/config로 학습 전 모델을 다시 만들어 before 샘플을 생성합니다."""
    set_seed(config.get("seed", GLOBAL_SEED))
    fresh_config = dict(config)
    fresh_config["actual_vocab_size"] = len(tokenizer.id_to_token)
    return LabGPTModel(model_config_from_experiment(fresh_config)).to(device)


def run_experiment(config: dict) -> dict:
    """토크나이저 준비, before 생성, 학습, after 생성, 결과 저장을 한 번에 수행합니다."""
    config = dict(config)
    print("=" * 80)
    print("RUN:", config["name"])
    tokenizer = train_or_load_tokenizer(
        vocab_size=config["tokenizer_vocab_size"],
        corpus=train_corpus,
        train_chars=config["tokenizer_train_chars"],
    )
    config["actual_vocab_size"] = len(tokenizer.id_to_token)

    before_model = build_fresh_lm_model(config, tokenizer)
    before_samples = generate_samples(before_model, tokenizer, config, PROMPTS, phase="before", decoding_names=["balanced"])
    del before_model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    train_loader, val_loader, token_stats = build_lm_loaders(config, tokenizer)
    print("token stats:", token_stats)
    model, history, summary = train_lm_model(config, tokenizer, train_loader, val_loader, token_stats)
    after_samples = generate_samples(model, tokenizer, config, PROMPTS, phase="after", decoding_names=list(GENERATION_PRESETS.keys()))
    samples = before_samples + after_samples

    run = {
        "config": dict(config),
        "tokenizer": tokenizer,
        "model": model,
        "history": history,
        "summary": summary,
        "samples": samples,
    }
    RUNS.append(run)
    save_lm_runs(RUNS)
    return run


def run_many(experiments: list[dict]) -> list[dict]:
    """여러 실험을 순서대로 실행하고 매 run 후 중간 결과를 표시합니다."""
    display_experiment_manifest(experiments)
    new_runs = []
    for config in experiments:
        run = run_experiment(config)
        new_runs.append(run)
        display_lm_summary(RUNS)
        plot_lm_histories(RUNS)
    return new_runs


def display_generation_comparison(runs: list[dict]) -> None:
    rows = generation_rows(runs)
    display_table(rows, [("run", "run"), ("phase", "phase"), ("decoding", "decoding"), ("prompt", "prompt"), ("text", "generated text")], title="생성문 비교")


def select_best_lm_run(runs: list[dict] | None = None) -> dict | None:
    """validation loss가 가장 낮은 LM run을 반환합니다."""
    runs = RUNS if runs is None else runs
    if not runs:
        return None
    return min(runs, key=lambda r: r["summary"]["best_val_loss"])


## 11. 사전학습 실험 실행

**목적**
- `EXPERIMENTS`에 정의한 사전학습 실험을 실행합니다.

**수정할 곳**
- 실행하려면 `new_runs = run_many(EXPERIMENTS)`의 주석을 해제합니다.

**출력**
- 실험 manifest
- run별 학습 로그
- 중간 요약표와 loss 그래프

**해석**
- manifest에서 의도한 변인만 바뀌었는지 먼저 확인합니다.
- `SMOKE`가 끝까지 돌면 `LIGHT` 이상으로 확장합니다.


In [ ]:
display_experiment_manifest(EXPERIMENTS)

# 실행하려면 아래 줄의 주석을 해제하세요.
# new_runs = run_many(EXPERIMENTS)

print("준비된 실험 수:", len(EXPERIMENTS))
print("실행 명령: new_runs = run_many(EXPERIMENTS)")


## 12. 사전학습 결과 시각화

**목적**
- 실행된 LM 실험을 표, 그래프, 생성문 비교로 확인합니다.

**수정할 곳**
- 보통 수정하지 않습니다.

**출력**
- 요약표
- loss/perplexity/gap/efficiency 그래프
- before/after 생성문 비교표

**해석**
- `delta`가 음수면 baseline보다 val loss가 낮아진 것입니다.
- gap이 커지면 과적합 가능성이 있습니다.
- runtime 대비 loss가 좋은 모델이 실험 효율이 좋습니다.


In [ ]:
display_lm_summary(RUNS)
plot_lm_histories(RUNS)
plot_lm_perplexity(RUNS)
plot_lm_gap(RUNS)
plot_efficiency_scatter(RUNS)
plot_runtime_scatter(RUNS)
display_generation_comparison(RUNS)
export_lm_artifacts(RUNS)


## 12.1 직접 입력 Playground

**목적**
- 학습 전/후 모델에 직접 문장을 넣어 LLM처럼 생성 결과를 확인합니다.

**수정할 곳**
- `PLAYGROUND_RUN_INDEX`: 사용할 run 번호
- `PLAYGROUND_PHASE`: `before` 또는 `after`
- `PLAYGROUND_PROMPT`: 입력 문장
- `PLAYGROUND_DECODING`: `greedy`, `balanced`, `creative`

**출력**
- 선택한 모델과 decoding 설정
- 생성 텍스트

**해석**
- greedy는 안정적이지만 단조로울 수 있습니다.
- creative는 다양하지만 품질이 흔들릴 수 있습니다.


In [ ]:
PLAYGROUND_RUN_INDEX = 0  # @param {type:"integer"}
PLAYGROUND_PHASE = "after"  # @param ["before", "after"]
PLAYGROUND_PROMPT = "이 영화는"  # @param {type:"string"}
PLAYGROUND_DECODING = "balanced"  # @param ["greedy", "balanced", "creative"]


def run_playground(run_index: int, phase: str, prompt: str, decoding: str) -> None:
    """선택한 run의 before/after 모델로 직접 입력 생성 결과를 표시합니다."""
    if not RUNS:
        display(Markdown("_먼저 사전학습 실험을 실행하세요._"))
        return
    if run_index < 0 or run_index >= len(RUNS):
        raise IndexError(f"run_index must be between 0 and {len(RUNS) - 1}")
    run = RUNS[run_index]
    config = run["config"]
    tokenizer = run["tokenizer"]
    if phase == "before":
        model = build_fresh_lm_model(config, tokenizer)
    else:
        model = run["model"]
    text = generate_text(model, tokenizer, prompt, config, decoding=decoding)
    if phase == "before":
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
    display_table([{
        "run": run["summary"]["name"],
        "phase": phase,
        "decoding": decoding,
        "prompt": prompt,
        "text": text,
    }], [("run", "run"), ("phase", "phase"), ("decoding", "decoding"), ("prompt", "prompt"), ("text", "generated text")], title="Playground 결과")


run_playground(PLAYGROUND_RUN_INDEX, PLAYGROUND_PHASE, PLAYGROUND_PROMPT, PLAYGROUND_DECODING)


## 13. 실험 예시 모음

**목적**
- 변인 통제 실험을 바로 실행할 수 있는 예시 목록을 제공합니다.

**수정할 곳**
- 원하는 예시 리스트를 `run_many(...)`에 넣어 실행합니다.
- 필요하면 일부 설정만 복사해 `EXPERIMENTS`에 붙입니다.

**출력**
- 예시 실험 리스트 이름

**해석**
- position → lr → activation → size → optimizer 순서로 넓혀가면 해석이 쉽습니다.


In [ ]:
# learning rate 비교
LR_EXPERIMENTS = [
    make_experiment("lr_1e-4", {"learning_rate": 1e-4}),
    make_experiment("lr_3e-4", {"learning_rate": 3e-4}),
    make_experiment("lr_5e-4", {"learning_rate": 5e-4}),
]

# 모델 크기 비교
SIZE_EXPERIMENTS = [
    make_experiment("tiny_1layer_64d", {"emb_dim": 64, "n_heads": 4, "n_layers": 1}),
    make_experiment("base_2layer_96d", {"emb_dim": 96, "n_heads": 4, "n_layers": 2}),
    make_experiment("wide_2layer_128d", {"emb_dim": 128, "n_heads": 4, "n_layers": 2}),
]

# 위치 임베딩 비교
POSITION_EXPERIMENTS = [
    make_experiment("pos_learned", {"position_encoding": "learned"}),
    make_experiment("pos_sinusoidal", {"position_encoding": "sinusoidal"}),
    make_experiment("pos_none", {"position_encoding": "none"}),
]

# 활성화 함수 비교
ACTIVATION_EXPERIMENTS = [
    make_experiment("act_gelu", {"activation": "gelu"}),
    make_experiment("act_relu", {"activation": "relu"}),
    make_experiment("act_silu", {"activation": "silu"}),
]

# optimizer/scheduler 비교
OPTIMIZER_EXPERIMENTS = [
    make_experiment("adamw", {"optimizer": "adamw", "scheduler": "none"}),
    make_experiment("adam", {"optimizer": "adam", "scheduler": "none", "weight_decay": 0.0}),
    make_experiment("adamw_cosine", {"optimizer": "adamw", "scheduler": "cosine_warmup", "warmup_steps": 20}),
]

# seed 안정성 비교
SEED_EXPERIMENTS = [
    make_experiment("seed_42", {"seed": 42}),
    make_experiment("seed_43", {"seed": 43}),
    make_experiment("seed_44", {"seed": 44}),
]

# 파인튜닝 범위 비교는 15번 셀의 FINETUNE_SCOPE_EXPERIMENTS를 사용합니다.
print("예시 리스트: LR_EXPERIMENTS, SIZE_EXPERIMENTS, POSITION_EXPERIMENTS, ACTIVATION_EXPERIMENTS, OPTIMIZER_EXPERIMENTS, SEED_EXPERIMENTS")
print("실행 예: run_many(POSITION_EXPERIMENTS)")


## 14. 파인튜닝 데이터와 freeze mode

**목적**
- 감성분류 데이터로 fine-tuning을 준비하고 평가 지표를 계산합니다.

**수정할 곳**
- 보통 15번 셀의 `BASE_FINETUNE` 값을 수정합니다.

**출력**
- DataLoader
- confusion matrix
- precision/recall/F1
- confidence 높은 오답 예시

**해석**
- `classifier_only`: 빠르고 안정적입니다.
- `last_block_plus_head`: 일부 backbone 적응을 허용합니다.
- `full_finetune`: 가장 강하지만 작은 데이터에서 과적합될 수 있습니다.


In [ ]:
def load_jsonl_rows(path: Path, limit: int | None = None, seed: int = GLOBAL_SEED) -> list[dict]:
    """JSONL 감성분류 데이터를 읽고 seed 기준으로 섞은 뒤 일부만 사용합니다."""
    rows = []
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    rng = random.Random(seed)
    rng.shuffle(rows)
    return rows[:limit] if limit is not None else rows


def build_sentiment_loaders(tokenizer: BPETokenizer, config: dict):
    """fine-tuning train/val/test DataLoader를 만듭니다."""
    train_rows = load_jsonl_rows(SENTIMENT_TRAIN_PATH, config.get("finetune_train_samples"), seed=config.get("seed", GLOBAL_SEED))
    val_rows = load_jsonl_rows(SENTIMENT_VAL_PATH, config.get("finetune_val_samples"), seed=config.get("seed", GLOBAL_SEED))
    test_rows = load_jsonl_rows(SENTIMENT_TEST_PATH, config.get("finetune_test_samples"), seed=config.get("seed", GLOBAL_SEED))

    max_length = min(config.get("finetune_max_length", config["context_length"]), config["context_length"])
    train_ds = ReviewSentimentDataset(train_rows, tokenizer, max_length=max_length)
    val_ds = ReviewSentimentDataset(val_rows, tokenizer, max_length=max_length)
    test_ds = ReviewSentimentDataset(test_rows, tokenizer, max_length=max_length)

    batch_size = config.get("finetune_batch_size", config["batch_size"])
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False),
        {"train_rows": train_rows, "val_rows": val_rows, "test_rows": test_rows},
    )


def apply_freeze_mode(model: GPTForSequenceClassification, freeze_mode: str) -> None:
    """freeze_mode에 따라 backbone의 학습 범위를 조절합니다."""
    for p in model.gpt.parameters():
        p.requires_grad = False

    if freeze_mode == "classifier_only":
        pass
    elif freeze_mode == "last_block_plus_head":
        if len(model.gpt.blocks) > 0:
            for p in model.gpt.blocks[-1].parameters():
                p.requires_grad = True
        for p in model.gpt.final_norm.parameters():
            p.requires_grad = True
    elif freeze_mode == "full_finetune":
        for p in model.gpt.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f"Unknown freeze_mode: {freeze_mode}")

    for p in model.classifier.parameters():
        p.requires_grad = True


def build_finetune_optimizer(model: GPTForSequenceClassification, config: dict):
    """backbone과 classifier head에 다른 learning rate를 적용합니다."""
    head_params = []
    backbone_params = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("classifier") or name.startswith("dropout"):
            head_params.append(p)
        else:
            backbone_params.append(p)

    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": config.get("finetune_backbone_lr", 1e-5)})
    if head_params:
        groups.append({"params": head_params, "lr": config.get("finetune_head_lr", 1e-3)})
    return torch.optim.AdamW(groups, weight_decay=config.get("finetune_weight_decay", 0.01))


def collect_sentiment_predictions(model, loader, raw_rows: list[dict]) -> list[dict]:
    """prediction, confidence, 정답 여부를 row 단위로 수집합니다."""
    predictions = []
    model.eval()
    offset = 0
    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device).long()
            logits = model(input_ids)
            probs = torch.softmax(logits, dim=-1)
            confs, preds = torch.max(probs, dim=-1)
            batch_rows = raw_rows[offset:offset + labels.size(0)]
            offset += labels.size(0)
            for row, y, p, conf in zip(batch_rows, labels.cpu().tolist(), preds.cpu().tolist(), confs.cpu().tolist()):
                predictions.append({
                    "text": row.get("text", ""),
                    "label": int(y),
                    "pred": int(p),
                    "confidence": float(conf),
                    "correct": int(y == p),
                    "error_type": "correct" if y == p else f"true_{y}_pred_{p}",
                })
    return predictions


def classification_metrics(predictions: list[dict]) -> dict:
    """binary classification precision/recall/F1을 계산합니다."""
    tp = sum(1 for r in predictions if r["label"] == 1 and r["pred"] == 1)
    tn = sum(1 for r in predictions if r["label"] == 0 and r["pred"] == 0)
    fp = sum(1 for r in predictions if r["label"] == 0 and r["pred"] == 1)
    fn = sum(1 for r in predictions if r["label"] == 1 and r["pred"] == 0)
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    accuracy = (tp + tn) / max(1, tp + tn + fp + fn)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1, "tp": tp, "tn": tn, "fp": fp, "fn": fn}


def confusion_matrix_from_predictions(predictions: list[dict]) -> np.ndarray:
    matrix = np.zeros((2, 2), dtype=int)
    for row in predictions:
        y, p = row["label"], row["pred"]
        if y in (0, 1) and p in (0, 1):
            matrix[y, p] += 1
    return matrix


def plot_confusion_matrix(matrix: np.ndarray, title: str, save_name: str | None = None) -> None:
    plt.figure(figsize=(4, 4))
    plt.imshow(matrix, cmap="Blues")
    plt.xticks([0, 1], ["pred 0", "pred 1"])
    plt.yticks([0, 1], ["true 0", "true 1"])
    plt.title(title)
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(matrix[i, j]), ha="center", va="center", color="black")
    plt.colorbar()
    if save_name:
        save_current_figure(save_name)
    plt.show()


## 15. 파인튜닝 실행 함수

**목적**
- 사전학습 run을 backbone으로 사용해 감성분류 fine-tuning을 실행합니다.

**수정할 곳**
- `FINETUNE_EXPERIMENTS`: 비교할 freeze mode 목록
- `finetune_head_lr`, `finetune_backbone_lr`: head/backbone learning rate

**출력**
- finetune summary/history/prediction CSV
- accuracy 그래프와 confusion matrix

**해석**
- 기본 backbone은 best validation loss LM run입니다.
- full fine-tuning이 classifier-only보다 항상 좋지는 않습니다.


In [ ]:
BASE_FINETUNE = {
    "seed": GLOBAL_SEED,
    "freeze_mode": "classifier_only",       # classifier_only | last_block_plus_head | full_finetune
    "use_pretrained_backbone": True,
    "finetune_epochs": preset["finetune_epochs"],
    "finetune_train_samples": preset["finetune_train_samples"],
    "finetune_val_samples": preset["finetune_val_samples"],
    "finetune_test_samples": preset["finetune_val_samples"],
    "finetune_batch_size": preset["batch_size"],
    "finetune_max_length": preset["context_length"],
    "finetune_backbone_lr": 1e-5,
    "finetune_head_lr": 1e-3,
    "finetune_weight_decay": 0.01,
}


def make_finetune_experiment(name: str, overrides: dict | None = None) -> dict:
    """BASE_FINETUNE에서 필요한 값만 덮어써 fine-tuning run 설정을 만듭니다."""
    config = dict(BASE_FINETUNE)
    config["name"] = name
    if overrides:
        config.update(overrides)
    return config


FINETUNE_SCOPE_EXPERIMENTS = [
    make_finetune_experiment("ft_classifier_only", {"freeze_mode": "classifier_only"}),
    make_finetune_experiment("ft_last_block", {"freeze_mode": "last_block_plus_head", "finetune_backbone_lr": 2e-5}),
    make_finetune_experiment("ft_full_finetune", {"freeze_mode": "full_finetune", "finetune_backbone_lr": 3e-5}),
]
FINETUNE_EXPERIMENTS = FINETUNE_SCOPE_EXPERIMENTS[:1] + FINETUNE_SCOPE_EXPERIMENTS[2:]


def run_finetune_experiment(config: dict, source_run: dict | None = None) -> dict:
    """하나의 fine-tuning 실험을 실행하고 metric과 prediction을 저장합니다."""
    set_seed(config.get("seed", GLOBAL_SEED))

    if source_run is None and config.get("use_pretrained_backbone", True):
        source_run = select_best_lm_run(RUNS)

    if source_run is not None and config.get("use_pretrained_backbone", True):
        tokenizer = source_run["tokenizer"]
        backbone = copy.deepcopy(source_run["model"])
        base_name = source_run["summary"]["name"]
    else:
        base_lm_config = make_experiment("scratch_for_finetune")
        tokenizer = train_or_load_tokenizer(base_lm_config["tokenizer_vocab_size"], train_corpus, base_lm_config["tokenizer_train_chars"])
        base_lm_config["actual_vocab_size"] = len(tokenizer.id_to_token)
        backbone = LabGPTModel(model_config_from_experiment(base_lm_config))
        base_name = "scratch"

    clf = GPTForSequenceClassification(backbone, num_labels=2, drop_rate=0.1).to(device)
    apply_freeze_mode(clf, config["freeze_mode"])
    optimizer = build_finetune_optimizer(clf, config)

    lm_context = clf.gpt.config["context_length"]
    loader_config = {
        **BASE_EXPERIMENT,
        **config,
        "context_length": lm_context,
        "batch_size": config.get("finetune_batch_size", BASE_EXPERIMENT["batch_size"]),
    }
    train_loader, val_loader, test_loader, raw_rows = build_sentiment_loaders(tokenizer, loader_config)

    history = []
    started = time.time()
    for epoch in range(1, int(config["finetune_epochs"]) + 1):
        train_loss, train_acc = train_epoch_sentiment(clf, train_loader, optimizer, device)
        val_loss, val_acc = evaluate_sentiment(clf, val_loader, device)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "elapsed_sec": time.time() - started,
        })
        print(f"[{config['name']}] epoch {epoch} | train acc {train_acc:.3f} | val acc {val_acc:.3f}")

    test_loss, test_acc = evaluate_sentiment(clf, test_loader, device)
    predictions = collect_sentiment_predictions(clf, test_loader, raw_rows["test_rows"])
    metrics = classification_metrics(predictions)
    matrix = confusion_matrix_from_predictions(predictions)
    high_conf_errors = sorted([r for r in predictions if not r["correct"]], key=lambda r: r["confidence"], reverse=True)[:10]

    summary = {
        "name": config["name"],
        "base_run": base_name,
        "freeze_mode": config["freeze_mode"],
        "trainable_params": count_parameters(clf, trainable_only=True),
        "total_params": count_parameters(clf),
        "best_val_acc": max(h["val_acc"] for h in history),
        "final_val_acc": history[-1]["val_acc"],
        "test_acc": test_acc,
        "test_loss": test_loss,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "runtime": format_seconds(time.time() - started),
    }
    run = {
        "config": dict(config),
        "model": clf,
        "history": history,
        "summary": summary,
        "confusion_matrix": matrix.tolist(),
        "predictions": predictions,
        "high_conf_errors": high_conf_errors,
    }
    FINETUNE_RUNS.append(run)
    export_finetune_artifacts(FINETUNE_RUNS)
    return run


def run_finetune_many(configs: list[dict], source_run: dict | None = None) -> list[dict]:
    """여러 fine-tuning 실험을 순서대로 실행합니다."""
    if source_run is None:
        source_run = select_best_lm_run(RUNS)
    runs = []
    for config in configs:
        runs.append(run_finetune_experiment(config, source_run=source_run))
    return runs


def finetune_summary_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        s = run["summary"]
        rows.append({
            "name": s["name"],
            "base": s["base_run"],
            "freeze": s["freeze_mode"],
            "trainable_params": s["trainable_params"],
            "best_val_acc": s["best_val_acc"],
            "test_acc": s["test_acc"],
            "precision": s["precision"],
            "recall": s["recall"],
            "f1": s["f1"],
            "fp": s["fp"],
            "fn": s["fn"],
            "runtime": s["runtime"],
        })
    return rows


def finetune_history_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        for h in run.get("history", []):
            rows.append({"run": run["summary"]["name"], **h})
    return rows


def finetune_prediction_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        for pred in run.get("predictions", []):
            rows.append({"run": run["summary"]["name"], **pred})
    return rows


def export_finetune_artifacts(runs: list[dict]) -> None:
    """fine-tuning 결과를 JSONL과 CSV로 저장합니다."""
    serializable = [{"summary": r["summary"], "history": r["history"], "config": r["config"], "confusion_matrix": r["confusion_matrix"], "high_conf_errors": r["high_conf_errors"]} for r in runs]
    save_jsonl(ARTIFACT_DIR / "finetune_runs.jsonl", serializable)
    write_csv(ARTIFACT_DIR / "finetune_summary.csv", finetune_summary_rows(runs))
    write_csv(ARTIFACT_DIR / "finetune_history.csv", finetune_history_rows(runs))
    write_csv(ARTIFACT_DIR / "finetune_predictions.csv", finetune_prediction_rows(runs))
    print("saved finetune artifacts:", ARTIFACT_DIR)


def display_finetune_summary(runs: list[dict]) -> None:
    rows = []
    for row in finetune_summary_rows(runs):
        rows.append({
            "name": row["name"],
            "base": row["base"],
            "freeze": row["freeze"],
            "trainable": f"{row['trainable_params']:,}",
            "best_val_acc": f"{row['best_val_acc']:.3f}",
            "test_acc": f"{row['test_acc']:.3f}",
            "f1": f"{row['f1']:.3f}",
            "fp/fn": f"{row['fp']}/{row['fn']}",
            "runtime": row["runtime"],
        })
    display_table(rows, [("name", "run"), ("base", "base"), ("freeze", "freeze"), ("trainable", "trainable params"), ("best_val_acc", "best val acc"), ("test_acc", "test acc"), ("f1", "F1"), ("fp/fn", "FP/FN"), ("runtime", "runtime")], title="파인튜닝 요약")


def plot_finetune_accuracy(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No finetune runs to plot.")
        return
    labels = [r["summary"]["name"] for r in runs]
    val_acc = [r["summary"]["best_val_acc"] for r in runs]
    test_acc = [r["summary"]["test_acc"] for r in runs]
    f1 = [r["summary"]["f1"] for r in runs]
    x = np.arange(len(labels))
    width = 0.25
    plt.figure(figsize=(10, 4))
    plt.bar(x - width, val_acc, width, label="best val acc")
    plt.bar(x, test_acc, width, label="test acc")
    plt.bar(x + width, f1, width, label="F1")
    plt.xticks(x, labels, rotation=20, ha="right")
    plt.ylim(0, 1)
    plt.ylabel("score")
    plt.title("Finetuning score comparison")
    plt.legend()
    plt.grid(axis="y", alpha=0.25)
    if save:
        save_current_figure("finetune_scores.png")
    plt.show()


def display_high_confidence_errors(runs: list[dict], limit: int = 10) -> None:
    rows = []
    for run in runs:
        for row in run.get("high_conf_errors", [])[:limit]:
            rows.append({
                "run": run["summary"]["name"],
                "label": row["label"],
                "pred": row["pred"],
                "confidence": f"{row['confidence']:.3f}",
                "text": row["text"],
            })
    display_table(rows, [("run", "run"), ("label", "label"), ("pred", "pred"), ("confidence", "confidence"), ("text", "text")], title="confidence 높은 오답")


## 16. 파인튜닝 비교 실행

**목적**
- `FINETUNE_EXPERIMENTS`에 정의된 감성분류 실험을 실행합니다.

**수정할 곳**
- 실행하려면 `ft_runs = run_finetune_many(FINETUNE_EXPERIMENTS)`의 주석을 해제합니다.
- 세 가지 freeze mode를 모두 비교하려면 `FINETUNE_SCOPE_EXPERIMENTS`를 실행합니다.

**출력**
- epoch별 train/val accuracy 로그
- finetune CSV 파일

**해석**
- best LM run이 자동으로 backbone으로 선택됩니다.
- full fine-tuning이 더 느린 만큼 성능 개선이 있는지 확인합니다.


In [ ]:
# 실행하려면 아래 줄의 주석을 해제하세요.
# ft_runs = run_finetune_many(FINETUNE_EXPERIMENTS)
# ft_scope_runs = run_finetune_many(FINETUNE_SCOPE_EXPERIMENTS)

best_lm = select_best_lm_run(RUNS)
print("준비된 파인튜닝 실험 수:", len(FINETUNE_EXPERIMENTS))
print("기본 backbone:", best_lm["summary"]["name"] if best_lm else "scratch 또는 LM run 없음")
print("실행 명령: ft_runs = run_finetune_many(FINETUNE_EXPERIMENTS)")


## 17. 파인튜닝 결과 시각화

**목적**
- fine-tuning 결과를 score, confusion matrix, 오답 예시로 확인합니다.

**수정할 곳**
- 보통 수정하지 않습니다.

**출력**
- accuracy/F1 그래프
- confusion matrix PNG
- confidence 높은 오답 표

**해석**
- FP가 많으면 부정을 긍정으로 잘못 예측하는 경향입니다.
- FN이 많으면 긍정을 부정으로 잘못 예측하는 경향입니다.


In [ ]:
display_finetune_summary(FINETUNE_RUNS)
plot_finetune_accuracy(FINETUNE_RUNS)
for run in FINETUNE_RUNS:
    plot_confusion_matrix(np.array(run["confusion_matrix"]), run["summary"]["name"], save_name=f"confusion_{run['summary']['name']}.png")
display_high_confidence_errors(FINETUNE_RUNS)
export_finetune_artifacts(FINETUNE_RUNS)


## 18. 보고서용 요약 생성

**목적**
- 실행 결과를 보고서에 바로 옮길 수 있는 Markdown으로 정리합니다.

**수정할 곳**
- 보통 수정하지 않습니다.

**출력**
- `lab_report_summary.md`
- 핵심 설정표, metric 표, 생성문 비교, 다음 실험 후보

**해석**
- best run만 보지 말고 baseline 대비 delta와 비용을 함께 봅니다.
- 생성문은 정성 평가이며, loss/accuracy와 함께 해석합니다.


In [ ]:
def markdown_table(rows: list[dict], columns: list[tuple[str, str]]) -> list[str]:
    """보고서용 Markdown 표 문자열 리스트를 만듭니다."""
    if not rows:
        return ["_결과 없음_"]
    header = "| " + " | ".join(label for _, label in columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(md_escape(row.get(key, "")) for key, _ in columns) + " |")
    return [header, sep] + body


def next_experiment_suggestions(lm_runs: list[dict], ft_runs: list[dict]) -> list[str]:
    """현재 결과를 바탕으로 다음 실험 후보를 간단히 제안합니다."""
    suggestions = []
    if lm_runs:
        best = min(lm_runs, key=lambda r: r["summary"]["best_val_loss"])
        suggestions.append(f"`{best['summary']['name']}` 주변 learning rate를 더 촘촘히 탐색")
        suggestions.append("best model size를 고정한 뒤 position/activation만 재비교")
        suggestions.append("같은 설정을 seed 42/43/44로 반복해 안정성 확인")
    else:
        suggestions.append("먼저 `SMOKE` preset으로 baseline과 position 비교 실행")
    if ft_runs:
        best_ft = max(ft_runs, key=lambda r: r["summary"]["best_val_acc"])
        suggestions.append(f"fine-tuning은 `{best_ft['summary']['freeze_mode']}` 기준으로 lr 재탐색")
    else:
        suggestions.append("best LM run으로 classifier_only/full_finetune 비교")
    return suggestions


def build_report_markdown(lm_runs: list[dict], ft_runs: list[dict]) -> str:
    lines = []
    lines.append("## lab.ipynb 실험 요약")
    lines.append("")
    lines.append(f"- preset: `{RUN_PRESET}`")
    lines.append(f"- artifact dir: `{ARTIFACT_DIR}`")
    lines.append("")

    if lm_runs:
        lines.append("### 사전학습 비교")
        lm_rows = []
        for row in lm_summary_rows(lm_runs):
            lm_rows.append({
                "run": row["name"],
                "position": row["position"],
                "activation": row["activation"],
                "best_val_loss": f"{row['best_val_loss']:.3f}",
                "delta": f"{row['delta_vs_baseline']:+.3f}",
                "ppl": f"{row['best_ppl']:.1f}",
                "params": f"{row['params']:,}",
                "runtime": row["runtime"],
            })
        lines.extend(markdown_table(lm_rows, [("run", "run"), ("position", "position"), ("activation", "activation"), ("best_val_loss", "best val loss"), ("delta", "delta"), ("ppl", "ppl"), ("params", "params"), ("runtime", "runtime")]))
        lines.append("")

        best_lm = min(lm_runs, key=lambda r: r["summary"]["best_val_loss"])
        s = best_lm["summary"]
        lines.append(f"- best LM run: `{s['name']}` / val loss `{s['best_val_loss']:.3f}` / ppl `{perplexity(s['best_val_loss']):.1f}`")
        lines.append("")
        lines.append("### 생성 샘플")
        sample_rows = []
        for sample in best_lm.get("samples", [])[:12]:
            sample_rows.append({"phase": sample.get("phase"), "decoding": sample.get("decoding"), "prompt": sample.get("prompt"), "text": sample.get("text")})
        lines.extend(markdown_table(sample_rows, [("phase", "phase"), ("decoding", "decoding"), ("prompt", "prompt"), ("text", "generated text")]))
        lines.append("")

    if ft_runs:
        lines.append("### 파인튜닝 비교")
        ft_rows = []
        for row in finetune_summary_rows(ft_runs):
            ft_rows.append({
                "run": row["name"],
                "freeze": row["freeze"],
                "val_acc": f"{row['best_val_acc']:.3f}",
                "test_acc": f"{row['test_acc']:.3f}",
                "f1": f"{row['f1']:.3f}",
                "fp/fn": f"{row['fp']}/{row['fn']}",
            })
        lines.extend(markdown_table(ft_rows, [("run", "run"), ("freeze", "freeze"), ("val_acc", "best val acc"), ("test_acc", "test acc"), ("f1", "F1"), ("fp/fn", "FP/FN")]))
        lines.append("")

    lines.append("### 다음 실험 후보")
    for item in next_experiment_suggestions(lm_runs, ft_runs):
        lines.append(f"- {item}")
    return chr(10).join(lines)


report_md = build_report_markdown(RUNS, FINETUNE_RUNS)
display(Markdown(report_md))
report_path = ARTIFACT_DIR / "lab_report_summary.md"
report_path.write_text(report_md, encoding="utf-8")
print("saved:", report_path)
